In [1]:
# this notebook will normalize the data
# while keeping key level as feature
import pandas as pd
import json
import numpy as np
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

symbol = "BTCUSDT"
tf = "5m"

folder_path = find_project_root() / "data" / "mlData" 
src_path = folder_path / "raw" / f"{symbol}-{tf}-vX.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
print(f"5m Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 20.22 MB
df.head()

5m Shape: (546454, 10) | Memory: 43.72 MB


,timestamp,open,high,low,close,vol,taker#,taker-buy-basevol,atr42,label
0,1609187400000,26852.179688,26873.220703,26688.109375,26736.640625,598.524536,9027,199.124802,NaN,NaN
1,1609187700000,26735.859375,26796.000000,26678.000000,26763.359375,527.531250,9375,215.260605,NaN,NaN
2,1609188000000,26763.359375,26824.490234,26733.599609,26779.490234,320.595581,5462,131.724228,NaN,NaN
3,1609188300000,26779.480469,26799.919922,26702.000000,26744.679688,254.103180,5055,108.755074,NaN,NaN
4,1609188600000,26744.970703,26861.980469,26744.970703,26850.050781,243.299011,4200,131.684021,NaN,NaN


In [2]:
# test : where is the train-test split boundary
# let's preview dataset characteristic
def describe_split(df, train_ratio=0.70, val_ratio=0.15):
    n       = len(df)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train = df.iloc[:n_train]
    val   = df.iloc[n_train : n_train + n_val]
    test  = df.iloc[n_train + n_val:]

    for name, split in [("train", train), ("val", val), ("test", test)]:
        print(f"{name:5s} | "
              f"from {split.index[0]}  to {split.index[-1]} | "
              f"rows: {len(split):,} | "
              f"close range: {split['close'].min():.0f} – {split['close'].max():.0f}")

describe_split(df, 0.70, 0.15)
describe_split(df, 0.80, 0.10)

train | from 0  to 382516 | rows: 382,517 | close range: 15594 – 73628
val   | from 382517  to 464484 | rows: 81,968 | close range: 52738 – 111924
test  | from 464485  to 546453 | rows: 81,969 | close range: 60082 – 126011
train | from 0  to 437162 | rows: 437,163 | close range: 15594 – 108985
val   | from 437163  to 491807 | rows: 54,645 | close range: 74610 – 124243
test  | from 491808  to 546453 | rows: 54,646 | close range: 60082 – 126011


# Be aware that ATR42 may created mechanistic shortcut 
- if MOM_3 > threshold → predict hit
- since the label window was also measured by ATR42

# Add features : Group I Rate of Change
see [Group I](../../ablations/README.md#group-i--rate-of-change)
- ROC_3
- ROC_5
- ROC_10
- MOM_3
- RETURNS_1

In [3]:
# group I Rate of Change feature
def compute_roc_features(df):
    # df has columns: close
    # df['close']
    
    # 1. rate of change 3 bars
    close_t_3 = df['close'].shift(3).bfill() # backward filled NaN
    df['ROC_3'] = (df['close']/close_t_3) - 1
    
    # 2. rate of change 5 bars
    close_t_5 = df['close'].shift(5).bfill() # backward filled NaN
    df['ROC_5'] = (df['close']/close_t_5) - 1
    
    # 2. rate of change 10 bars
    close_t_10 = df['close'].shift(10).bfill() # backward filled NaN
    df['ROC_10'] = (df['close']/close_t_10) - 1
    
    # 4. 3 bars momentum - normalized by [ATR-label window]
    # MOM_3 = (close_t - close_{t-3}) / ATR_42
    atr = df['atr42'].bfill() # backward fill from last atr
    df['MOM_3'] = (df['close']-close_t_3)/atr

    # 5. Return compared to previous bar %
    # Note Log return won't help much here since outlier
    #  + 20% in 15 mins was improbalble, and we will standardize later anyway
    df['RETURNS_1'] = df['close'].pct_change()
    
    return df

# Add features : Group L Orderflow
see [Group L](../../ablations/README.md#group-l--order-flow-imbalance)
- DELTA_1
- DELTA_3
- BUY_RATIO
- VOL_SPIKE
- DELTA_DIV

In [4]:
# group L orderflow feature
def compute_orderflow_features(df):
    # df has columns: volume, taker_buy_vol (V field)
    # df['vol'], df['taker-buy-basevol']
    
    # calculate sell volume
    sell_vol = df['vol'] - df['taker-buy-basevol']
    
    # 1. Per-bar Volume Delta
    df['DELTA_1'] = df['taker-buy-basevol'] - sell_vol
    
    # 2. Volume imbalance ratio
    df['BUY_RATIO'] = df['taker-buy-basevol'] / df['vol']
    
    # 3. Cumulative delta (last N bars) 3-10
    # fillNa with 0
    df['DELTA_3'] = df['DELTA_1'].rolling(3).sum()

    # 4. Volume spiked
    sma_vol_5 = df['vol'].rolling(5).mean()
    df['VOL_SPIKE'] = df['vol'] / (sma_vol_5)

    # 5. Delta divergence (price up but delta down = exhaustion)
    # binary
    sign_roc = np.sign(df['ROC_3'])
    sign_delta = np.sign(df['DELTA_3'])
    df['DELTA_DIV'] = (sign_roc != sign_delta).astype(int)
    
    
    return df

# Add group J
```
ATR_5	RMA(Abs(high−low),5)	5 bars
ATR_14	RMA(Abs(high−low),14)	14 bars
ATR_RATIO	ATR_5 / ATR_42	5 / 42 bars
ATR_NORM_ROC	ROC_3 / ATR_14	3 bars
RANGE_RATIO	(high − low) / ATR_14	1 bar
```

In [5]:
# group J ATR Volatility Ratio
# Using SMA for consistency with atr42 (computed in 1prepData)
# True Range = max(high-low, |high-prev_close|, |low-prev_close|)

def compute_volatility_features(df):
    # df has columns: high, low, close, atr42, ROC_3
    prev_close = df['close'].shift(1)
    true_range = pd.concat([
        (df['high'] - df['low']).abs(),
        (df['high'] - prev_close).abs(),
        (df['low']  - prev_close).abs(),
    ], axis=1).max(axis=1)

    # 1. ATR_5  = SMA(TR, 5)
    df['ATR_5'] = true_range.rolling(5).mean().astype('float32')

    # 2. ATR_14 = SMA(TR, 14)
    df['ATR_14'] = true_range.rolling(14).mean().astype('float32')

    # 3. ATR_RATIO = ATR_5 / ATR_42
    df['ATR_RATIO'] = df['ATR_5'] / df['atr42'].bfill()

    # 4. ATR_NORM_ROC = ROC_3 / ATR_14  (normalised momentum)
    df['ATR_NORM_ROC'] = df['ROC_3'] / df['ATR_14']

    # 5. RANGE_RATIO = (high - low) / ATR_14  (current bar width vs avg)
    # intentional
    df['RANGE_RATIO'] = (df['high'] - df['low']).abs() / df['ATR_14']

    return df

# Add group K
```
Feature	Formula	Lookback
RSI_5	Wilder RSI, period=5	5 bars
RSI_14	Wilder RSI, period=14	14 bars
RSI_SLOPE	RSI_14_t − RSI_14_{t-3}	3 bars
STOCH_K	(close − low_5) / (high_5 − low_5) × 100	5 bars
CCI_5	(close − SMA_5) / (0.015 × MAD_5)	5 bars
```

In [6]:
# group j
def compute_oscillators_features(df):
    # df has columns: close, high, low

    delta = df['close'].diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)

    # Wilder smoothing:
    # Uses a modified moving average (RMA/SMMA) that gives more weight to previous data, 
    # effectively smoothing the result over a longer period
    def wilder_rsi(gain, loss, period):
        avg_gain = gain.ewm(com=period - 1, adjust=False).mean() # RMA
        avg_loss = loss.ewm(com=period - 1, adjust=False).mean() # RMA
        rs = avg_gain / avg_loss.replace(0, np.nan)
        return (100 - (100 / (1 + rs))).astype('float32')

    # 1. RSI_5
    df['RSI_5']  = wilder_rsi(gain, loss, 5)

    # 2. RSI_14
    df['RSI_14'] = wilder_rsi(gain, loss, 14)

    # 3. RSI_SLOPE = RSI_14_t - RSI_14_{t-3}
    df['RSI_SLOPE'] = (df['RSI_14'] - df['RSI_14'].shift(3)).astype('float32')

    # 4. STOCH_K = (close - low_5) / (high_5 - low_5) * 100
    low_5  = df['low'].rolling(5).min()
    high_5 = df['high'].rolling(5).max()
    df['STOCH_K'] = ((df['close'] - low_5) / (high_5 - low_5).replace(0, np.nan) * 100).astype('float32')

    # 5. CCI_5 = (close - SMA_5) / (0.015 * MAD_5)
    sma_5 = df['close'].rolling(5).mean()
    mad_5 = df['close'].rolling(5).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    df['CCI_5'] = ((df['close'] - sma_5) / (0.015 * mad_5.replace(0, np.nan))).astype('float32')

    return df

# Add group M
```
Group M — Distance to Swing High / Low
Feature	Formula	Lookback
DIST_HIGH_5	(rolling_max(high, 5) − close) / ATR_14	5 bars
DIST_LOW_5	(close − rolling_min(low, 5)) / ATR_14	5 bars
DIST_HIGH_10	(rolling_max(high, 10) − close) / ATR_14	10 bars
DIST_LOW_10	(close − rolling_min(low, 10)) / ATR_14	10 bars
RANGE_POS	(close − low_10) / (high_10 − low_10)	10 bars
```

In [7]:
# group M Distance to Swing High / Low
def compute_distance_swing_features(df):
    # df has columns: high, low, close, ATR_14 (computed in group J)

    atr14 = df['ATR_14'].replace(0, np.nan)

    # 1. DIST_HIGH_5 = (rolling_max(high, 5) − close) / ATR_14
    df['DIST_HIGH_5'] = ((df['high'].rolling(5).max() - df['close']) / atr14).astype('float32')

    # 2. DIST_LOW_5 = (close − rolling_min(low, 5)) / ATR_14
    df['DIST_LOW_5'] = ((df['close'] - df['low'].rolling(5).min()) / atr14).astype('float32')

    # 3. DIST_HIGH_10 = (rolling_max(high, 10) − close) / ATR_14
    high_10 = df['high'].rolling(10).max()
    df['DIST_HIGH_10'] = ((high_10 - df['close']) / atr14).astype('float32')

    # 4. DIST_LOW_10 = (close − rolling_min(low, 10)) / ATR_14
    low_10 = df['low'].rolling(10).min()
    df['DIST_LOW_10'] = ((df['close'] - low_10) / atr14).astype('float32')

    # 5. RANGE_POS = (close − low_10) / (high_10 − low_10)   [0=bottom, 1=top]
    df['RANGE_POS'] = ((df['close'] - low_10) / (high_10 - low_10).replace(0, np.nan)).astype('float32')

    return df

In [8]:
df.head()

,timestamp,open,high,low,close,vol,taker#,taker-buy-basevol,atr42,label
0,1609187400000,26852.179688,26873.220703,26688.109375,26736.640625,598.524536,9027,199.124802,NaN,NaN
1,1609187700000,26735.859375,26796.000000,26678.000000,26763.359375,527.531250,9375,215.260605,NaN,NaN
2,1609188000000,26763.359375,26824.490234,26733.599609,26779.490234,320.595581,5462,131.724228,NaN,NaN
3,1609188300000,26779.480469,26799.919922,26702.000000,26744.679688,254.103180,5055,108.755074,NaN,NaN
4,1609188600000,26744.970703,26861.980469,26744.970703,26850.050781,243.299011,4200,131.684021,NaN,NaN


# Special Group N
- My take on Distance to Swing High/Low
- Super large now 38 MB for test

In [9]:
# # group M Distance to Swing High / Low
# def compute_ict_pivot_features(df):
#     # df has columns: 15STR_confirmed	45STR_confirmed	15STR_keylv_high	15STR_keylv_low	barsSince15STR	45STR_keylv_high	45STR_keylv_low	barsSince45STR
#     # and others from I-M

#     # STR_CONFIRMED - yes 15 but ternary -1, 0, 1
#     # ITR_CONFIRMED - yes 45 but also ternary
#     # BARSINCE_STR - yes 15 int
#     # BARSINCE_ITR - yes 45 int

#     df['DIST_HIGH_15STR'] = (df['15STR_keylv_high'] - df['close'])/df['atr42'].bfill()
#     df['DIST_LOW_15STR'] = (df['close']-df['15STR_keylv_low'])/df['atr42'].bfill()
#     df['DIST_HIGH_45STR'] = (df['45STR_keylv_high'] - df['close'])/df['atr42'].bfill()
#     df['DIST_LOW_45STR'] = (df['close']-df['45STR_keylv_low'])/df['atr42'].bfill()
#     df['RANGE_15STR'] = (df['close']-df['15STR_keylv_low'])/(df['15STR_keylv_high']-df['15STR_keylv_low'])
#     df['RANGE_45STR'] = (df['close']-df['45STR_keylv_low'])/(df['45STR_keylv_high']-df['45STR_keylv_low'])

#     return df

In [10]:
# Apply augmentation
df = compute_roc_features(df)                  # group I
df = compute_orderflow_features(df)            # group L
df = compute_volatility_features(df)           # group J
df = compute_oscillators_features(df)          # group K
df = compute_distance_swing_features(df)       # group M
# df = compute_ict_pivot_features(df)            # group N

print(f"5m Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/ 1e6:.2f} MB")
print(df.columns)
df.head()

5m Shape: (546454, 35) | Memory: 126.78 MB
Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'taker#',
       'taker-buy-basevol', 'atr42', 'label', 'ROC_3', 'ROC_5', 'ROC_10',
       'MOM_3', 'RETURNS_1', 'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE',
       'DELTA_DIV', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC',
       'RANGE_RATIO', 'RSI_5', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5',
       'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10',
       'RANGE_POS'],
      dtype='object')


,timestamp,open,high,low,close,vol,taker#,taker-buy-basevol,atr42,label,...,RSI_5,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609187400000,26852.179688,26873.220703,26688.109375,26736.640625,598.524536,9027,199.124802,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1609187700000,26735.859375,26796.000000,26678.000000,26763.359375,527.531250,9375,215.260605,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1609188000000,26763.359375,26824.490234,26733.599609,26779.490234,320.595581,5462,131.724228,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1609188300000,26779.480469,26799.919922,26702.000000,26744.679688,254.103180,5055,108.755074,NaN,NaN,...,73.868942,90.650436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1609188600000,26744.970703,26861.980469,26744.970703,26850.050781,243.299011,4200,131.684021,NaN,NaN,...,86.860428,92.834373,NaN,88.131424,156.969452,NaN,NaN,NaN,NaN,NaN


# Analyse what to drop
- Obvious non-staionary data
- Obvious leakage data

In [11]:
processed_cols = ['open', 'high', 'low', 'close', 'vol', 'taker#', 'taker-buy-basevol']
non_feature_cols = ['atr42']
# keylv columns are intermediate — used to compute DIST/RANGE in group N, not features themselves
# keylv_cols = ['15STR_keylv_high', '15STR_keylv_low','45STR_keylv_high', '45STR_keylv_low']
merged = [*processed_cols, *non_feature_cols]

df.drop(columns=merged, inplace=True)  # called once

In [ ]:
print(f"5m Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 91.80 MB
df.head()

5m Shape: (546454, 27) | Memory: 91.80 MB


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RSI_5,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609187400000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,-200.274933,0.332693,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1609187700000,NaN,0.000999,0.000999,0.000999,0.248526,0.000999,-97.010040,0.408053,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1609188000000,NaN,0.001603,0.001603,0.001603,0.398567,0.000603,-57.147125,0.410873,-354.432098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1609188300000,NaN,0.000301,0.000301,0.000301,0.074776,-0.001300,-36.593033,0.427996,-190.750198,...,73.868942,90.650436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1609188600000,NaN,0.003239,0.004242,0.004242,0.806364,0.003940,20.069031,0.541244,-73.671127,...,86.860428,92.834373,NaN,88.131424,156.969452,NaN,NaN,NaN,NaN,NaN


In [13]:
# save to JSONL for training
out_path = folder_path / "augmented" / f"{symbol}-{tf}-vX-augmented.jsonl"

# note: not drop NaN warmup here

df.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df.shape}")
print(f"Saved {len(df)} rows to {out_path}")

final shape : (546454, 27)
Saved 546454 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-candle-science/data/mlData/augmented/BTCUSDT-5m-vX-augmented.jsonl


In [14]:
df.head()

,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RSI_5,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609187400000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,-200.274933,0.332693,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1609187700000,NaN,0.000999,0.000999,0.000999,0.248526,0.000999,-97.010040,0.408053,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1609188000000,NaN,0.001603,0.001603,0.001603,0.398567,0.000603,-57.147125,0.410873,-354.432098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1609188300000,NaN,0.000301,0.000301,0.000301,0.074776,-0.001300,-36.593033,0.427996,-190.750198,...,73.868942,90.650436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1609188600000,NaN,0.003239,0.004242,0.004242,0.806364,0.003940,20.069031,0.541244,-73.671127,...,86.860428,92.834373,NaN,88.131424,156.969452,NaN,NaN,NaN,NaN,NaN
